# 03 - Object Detection & Bounding Boxes

This notebook draws bounding boxes on detected objects, compares results
at different confidence thresholds side by side, and counts objects by
class with a bar chart. Everything runs in demo mode with cached data.

## Setup

In [ ]:
# !pip install oci-vision-ai[notebooks]

from oci_vision.core.client import VisionClient
from oci_vision.core.renderer import render_overlay
from oci_vision.core.models import AnalysisReport, DetectionResult
from PIL import Image

client = VisionClient(demo=True)
print(f"Client ready (demo={client.is_demo})")

## Run the analysis

In [ ]:
# Detect objects in the demo image
result = client.detect_objects("dog_closeup.jpg")

print(f"Model version: {result.model_version}")
print(f"Objects found: {len(result.objects)}")
for obj in result.objects:
    print(f"  - {obj.name}: {obj.confidence_pct}%")

## Explore the results

### Detection table with bounding box details

Each detected object has a name, confidence score, and a bounding polygon
defined by normalized vertices (0-1 range).

In [ ]:
# Detailed detection table
print(f"{'#':>3s}  {'Object':8s}  {'Conf':>7s}  {'Position':10s}  {'BBox (x_min, y_min, x_max, y_max)'}")
print('-' * 75)
for i, obj in enumerate(result.objects, 1):
    verts = obj.bounding_polygon.normalized_vertices
    x_min = min(v.x for v in verts)
    y_min = min(v.y for v in verts)
    x_max = max(v.x for v in verts)
    y_max = max(v.y for v in verts)
    pos = obj.bounding_polygon.human_position(1024, 768)
    print(f"{i:3d}  {obj.name:8s}  {obj.confidence_pct:6.2f}%  {pos:10s}  "
          f"({x_min:.3f}, {y_min:.3f}, {x_max:.3f}, {y_max:.3f})")

### Compare confidence thresholds

In [ ]:
# Show how filtering changes results at different thresholds
thresholds = [0.0, 0.80, 0.95]
for t in thresholds:
    filtered = result.above_threshold(t) if t > 0 else result.objects
    names = [o.name for o in filtered]
    print(f"Threshold >= {t:.0%}: {len(filtered)} objects -> {', '.join(names)}")

### Object count by class

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

# Count objects by class name
counts = Counter(obj.name for obj in result.objects)
classes = list(counts.keys())
values = list(counts.values())

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653']
ax.bar(classes, values, color=colors[:len(classes)])
ax.set_ylabel('Count')
ax.set_title('Detected Objects by Class')
for i, (cls, val) in enumerate(zip(classes, values)):
    ax.text(i, val + 0.1, str(val), ha='center', fontweight='bold')
ax.set_ylim(0, max(values) + 1.5)
plt.tight_layout()
plt.savefig('detection_counts.png', dpi=100, bbox_inches='tight')
plt.show()

## Visualize

### Threshold comparison: All objects vs > 80% vs > 95%

We render three versions of the same image, each showing only objects
above a different confidence threshold.

In [ ]:
from oci_vision.gallery import get_gallery_path

img_path = get_gallery_path() / "images" / "dog_closeup.jpg"
img = Image.open(img_path)

def make_report_at_threshold(det_result, threshold):
    """Create an AnalysisReport with only objects above the threshold."""
    if threshold <= 0:
        filtered = det_result
    else:
        filtered = DetectionResult(
            model_version=det_result.model_version,
            objects=det_result.above_threshold(threshold)
        )
    return AnalysisReport(
        image_path="dog_closeup.jpg",
        detection=filtered
    )

# Build three overlays
overlays = []
for t_label, t_val in [('All objects', 0.0), ('>80%', 0.80), ('>95%', 0.95)]:
    rpt = make_report_at_threshold(result, t_val)
    n = len(rpt.detection.objects)
    ov = render_overlay(img, rpt)
    overlays.append((t_label, n, ov))

# Display side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (label, n, ov) in zip(axes, overlays):
    ax.imshow(ov)
    ax.set_title(f"{label} ({n} objects)")
    ax.axis('off')
plt.tight_layout()
plt.savefig('detection_threshold_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

### Full annotated overlay

In [ ]:
# Full overlay with all detections
full_report = client.analyze("dog_closeup.jpg", features=["detection"])
overlay = render_overlay(img, full_report)
overlay.save('detection_overlay.png')
print(f"Overlay: {overlay.size[0]}x{overlay.size[1]} with {len(full_report.detection.objects)} objects")
overlay

## Under the hood

In [ ]:
import json

raw = result.model_dump()
print(json.dumps(raw, indent=2, default=str)[:3000])
print('...')

## Try it yourself

1. **Bounding box arithmetic**: Use `bounding_polygon.to_pixels(w, h)` to
   convert normalized coordinates to pixel values for a specific image size.
2. **Custom threshold**: Find the threshold that best separates real objects
   from false positives in the demo data.
3. **Position analysis**: Group objects by `human_position()` to understand
   spatial distribution (e.g., all dogs are center-left).
4. **Live mode**: Use `VisionClient()` with credentials to detect objects
   in your own photos.